In [ ]:
import pandas as pd 
import os

In [ ]:
#Gene expression data
gene_exp = pd.read_csv("data/raw/BRCA_gene_exp",sep="\t",index_col=0)
print(gene_exp.shape)
print(gene_exp.head())
print(gene_exp.index[:5])
print(gene_exp.columns[:5])
print(gene_exp.dtypes)

In [ ]:
#DNA methylation Data
meth = pd.read_csv("data/raw/HumanMethylation27",sep="\t",index_col=0)
print(meth.shape)
print(meth.head())
print(meth.index[:5])
print(meth.columns[:5])
print(meth.dtypes)

In [ ]:
#Clinical Data
clin = pd.read_csv("data/raw/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix",sep="\t",index_col=0)
print(clin.shape)
print(clin.head())
print(clin.index[:5])
print(clin.columns[:5])
print(clin.dtypes)

### Cohort Integrity Checks


In [ ]:
#First five gene expression sample IDs
print("RNA:")
print(gene_exp.columns[:5].tolist())

# First five methylation sample IDs
print("Methylation:")
print(meth.columns[:5].tolist())

# First five clinical sample IDs
print("Clinical:")
print(clin.index[:5].tolist())

In [ ]:
print(gene_exp.columns.to_series().str.len().value_counts())
print(meth.columns.to_series().str.len().value_counts())
print(clin.index.to_series().str.len().value_counts())

In [ ]:
# Extract patient identifiers (first 12 characters)
rna_patients = gene_exp.columns.to_series().str[:12]
meth_patients = meth.columns.to_series().str[:12]
clinical_patients = clin.index.to_series().str[:12]

print("RNA unique patients:", rna_patients.nunique())
print("RNA duplicate patient samples:", rna_patients.duplicated().sum())

print("Methylation unique patients:", meth_patients.nunique())
print("Methylation duplicate patient samples:", meth_patients.duplicated().sum())

print("Clinical unique patients:", clinical_patients.nunique())
print("Clinical duplicate patient samples:", clinical_patients.duplicated().sum())

In [ ]:
#samples shared across all the three datasets
rna_samples = set(gene_exp.columns)
meth_sample=set(meth.columns)
clinical_samples=set(clin.index)

print("RNA and methylation:",len(rna_samples & meth_sample))
print("RNA and clinical:",len(rna_samples & clinical_samples))
print("Methylation and clinical:",len(meth_sample & clinical_samples))
print("shared across all three:", len(rna_samples&meth_sample&clinical_samples))


In [ ]:
print(clin["ER_Status_nature2012"].value_counts(dropna=False))

In [ ]:
print(clin.loc[clin.index.isin(rna_samples & meth_sample),"ER_Status_nature2012"].value_counts(dropna=False))

In [ ]:
#construct the labelled cohort
shared_samples = rna_samples & meth_sample & clinical_samples
clinical_shared = clin.loc[clin.index.isin(shared_samples)].copy()
clinical_clean = clinical_shared[clinical_shared["ER_Status_nature2012"].isin(["Positive","Negative"])].copy()
print(clinical_clean["ER_Status_nature2012"].value_counts())
print("Total sample:",len(clinical_clean))

In [ ]:
# Extract patient identifiers from the final cohort
final_patients = clinical_clean.index.to_series().str[:12]

print("Total samples:", len(final_patients))
print("Unique patients:", final_patients.nunique())
print("Additional samples:", final_patients.duplicated().sum())

# Identify patients appearing more than once
patient_counts = final_patients.value_counts()

print("Patients with multiple samples:")
print(patient_counts[patient_counts > 1])

In [ ]:
clinical_clean["patient_id"] = clinical_clean.index.str[:12]
print(clinical_clean.groupby("patient_id")["ER_Status_nature2012"].nunique().value_counts())

### Alignment of omics matrices

In [ ]:
sample_ids = clinical_clean.index 
x_rna = gene_exp.loc[:,sample_ids].T.copy()
x_meth = meth.loc[:,sample_ids].T.copy()
y = clinical_clean["ER_Status_nature2012"].map({"Positive":1,"Negative":0}) #known ER status

In [ ]:
print("RNA dimensions:",x_rna.shape)
print("Methylation dimensions",x_meth.shape)
print("Target dimensions:",y.shape)
print("RNA and methylation identical indices:",x_rna.index.equals(x_meth.index))
print("RNA and target identical indices:",x_rna.index.equals(y.index))
print("Methylation and target identical indices:",x_meth.index.equals(y.index))

### Quality control

#### Methylation

In [ ]:
#Check for missing values
print("RNA missing values:",x_rna.isna().sum().sum())
print("Methylation missing values:",x_meth.isna().sum().sum())
print("RNA data types:")
print(x_rna.dtypes.value_counts())
print("Methylation data types:")
print(x_meth.dtypes.value_counts())

In [ ]:
#Investigate where the missing values in methylation are
missing_per_sample = x_meth.isna().sum(axis=1)
print(missing_per_sample.describe())
print("Samples with most missing values:")
print(missing_per_sample.sort_values(ascending=False).head(10))

In [ ]:
missing_per_probe = x_meth.isna().sum(axis=0)
print(missing_per_probe.describe())
print("Probes with most missing values:")
print(missing_per_probe.sort_values(ascending=False).head(10))

In [ ]:
sample_missing = x_meth.isna().mean(axis=1)*100
probe_missing = x_meth.isna().mean(axis=0)*100
print("Samples with >20% missing:",(sample_missing >20).sum())
print("Probes with > 20% missing:",(probe_missing>20).sum())
print("Completely missing probes:",(probe_missing==100).sum())

In [ ]:
#Remove missing probes
# Remove CpG probes that are completely missing
x_meth_qc = x_meth.dropna(axis=1, how="all")

print("Original methylation dimensions:", x_meth.shape)
print("After removing completely missing probes:", x_meth_qc.shape)
print("Remaining missing values:",x_meth_qc.isna().sum().sum())

In [ ]:
# Missingness per patient after removing completely missing probes
remaining_missing = x_meth_qc.isna().mean(axis=1) * 100

print(remaining_missing.describe())
print("Patients with >10% missing:",(remaining_missing > 10).sum())


In [ ]:
#Partially missing probes
# Number of probes containing at least one missing value
print("Partially missing probes:",x_meth_qc.isna().any(axis=0).sum())

# Number of probes with more than 5% missingness
probe_missing_pct = x_meth_qc.isna().mean(axis=0) * 100

print("Probes with >5% missing:",(probe_missing_pct > 5).sum())

# Number of probes with more than 10% missingness
print("Probes with >10% missing:",(probe_missing_pct > 10).sum())

In [ ]:
# Filter the methylation matrix
probe_missing_pct = x_meth_qc.isna().mean(axis=0) * 100

# Retain probes with <=5% missingness
selected_probes = probe_missing_pct <= 5

x_meth_filtered = x_meth_qc.loc[:, selected_probes].copy()

print("Original matrix:", x_meth.shape)
print("After removing completely missing probes:", x_meth_qc.shape)
print("After 5% missingness filtering:", x_meth_filtered.shape)

print("Remaining missing values:",x_meth_filtered.isna().sum().sum())

In [ ]:
# check if missing values are distributed across samples or concentrated in a few probes
sample_missing = x_meth_filtered.isna().mean(axis=1) * 100

print(sample_missing.describe())
print("Samples with >1% missingness:",(sample_missing > 1).sum())

# Missingness per probe
probe_missing = x_meth_filtered.isna().mean(axis=0) * 100

print(probe_missing.describe())
print("Probes with any missing values:",(probe_missing > 0).sum())

In [ ]:
#sample with the highest missingness
sample_missing.sort_values(ascending=False).head()


In [ ]:
sample_missing.idxmax()

In [ ]:
#sample 'TCGA-C8-A134-01' inspection
sample_id = "TCGA-C8-A134-01"
missing_probes = x_meth_filtered.loc[sample_id].isna()

print("Missing probes:", missing_probes.sum())
print("Total probes:", len(missing_probes))
print("Missing percentage:", missing_probes.mean() * 100)
print(clinical_clean.loc[sample_id, "ER_Status_nature2012"])

#### RNA expresssion

In [ ]:
print("RNA dimensions:", x_rna.shape)
print("Missing values:", x_rna.isna().sum().sum())
print("Data types:", x_rna.dtypes.value_counts())
print("Expression summary:")
print(x_rna.describe().loc[["mean", "std", "min", "max"]].T.describe())

In [ ]:
gene_variance = x_rna.var(axis=0)
print("Zero-variance genes:", (gene_variance == 0).sum())

In [ ]:
gene_variance = x_rna.var(axis=0)
x_rna_filtered = x_rna.loc[:, gene_variance > 0].copy()
print("Original RNA:", x_rna.shape)
print("Filtered RNA:", x_rna_filtered.shape)

In [ ]:
gene_variance = x_rna.var(axis=0)
x_rna_filtered = x_rna.loc[:, gene_variance > 0].copy()

print(x_rna_filtered.shape)
print(x_meth_filtered.shape)
print(y.shape)

In [ ]:
print("RNA and methylation aligned:",x_rna_filtered.index.equals(x_meth_filtered.index))
print("RNA and target aligned:",x_rna_filtered.index.equals(y.index))
print("Methylation and target aligned:",x_meth_filtered.index.equals(y.index))
print("Duplicate samples:",x_rna_filtered.index.duplicated().sum())

In [ ]:
os.makedirs("processed",exist_ok=True)
x_rna_filtered.to_csv("processed/rna_exp_clean.csv.gz",compression="gzip")
x_meth_filtered.to_csv("processed/methylation_clean.csv.gz",compression="gzip")
y.to_csv("processed/er_status.csv")


## Model Training


### Training set and test set split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_ids,test_ids = train_test_split(y.index,test_size=0.20,random_state=42,stratify=y)
#split RNA
x_rna_train = x_rna_filtered.loc[train_ids]
x_rna_test = x_rna_filtered.loc[test_ids]

#split methylation
x_meth_train = x_meth_filtered.loc[train_ids]
x_meth_test = x_meth_filtered.loc[test_ids]

#split target
y_train = y.loc[train_ids]
y_test = y.loc[test_ids]


In [ ]:
print("Training:",x_rna_train.shape,y_train.shape)
print("Test:",x_rna_test.shape,y_test.shape)
print("Training class distribution:", y_train.value_counts())
print("Test class distribution:",y_test.value_counts())

### Build the RNA model


In [ ]:
#import the packages
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import VarianceThreshold

In [ ]:
rna_pipeline = Pipeline([("variance",VarianceThreshold()),
                         ("selection",SelectKBest(score_func=f_classif,k=100)),
                         ("scaling",StandardScaler()),
                         ("classifier",LogisticRegression(class_weight="balanced",max_iter=2000,random_state=42))])

In [ ]:
rna_pipeline.fit(x_rna_train,y_train) #train the model
y_pred_rna = rna_pipeline.predict(x_rna_test)
y_prob_rna = rna_pipeline.predict_proba(x_rna_test)[:,1]

In [ ]:
print(y_pred_rna[:10])
print(y_prob_rna[:10])

### Evaluate the RNA model


In [ ]:
from sklearn.metrics import (accuracy_score,precision_score,recall_score, 
                             f1_score, roc_auc_score,confusion_matrix,classification_report)

In [ ]:
print("Accuracy:",accuracy_score(y_test,y_pred_rna))
print("precision:",precision_score(y_test,y_pred_rna))
print("Recall:",recall_score(y_test,y_pred_rna))
print("F1-score:",f1_score(y_test,y_pred_rna))
print("ROC_AUC:",roc_auc_score(y_test,y_prob_rna))
print("\nConfusion matrix:",confusion_matrix(y_test,y_pred_rna))
print("\nClassification report:",classification_report(y_test,y_pred_rna))

### Build the methylation model

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
meth_pipeline = Pipeline([("imputer",SimpleImputer(strategy="median")),
                          ("variance",VarianceThreshold()),
                          ("selection",SelectKBest(score_func=f_classif,k=100)),
                          ("scaling",StandardScaler()),
                          ("classifier",LogisticRegression(class_weight="balanced",max_iter=2000,random_state=42))])

In [ ]:
meth_pipeline.fit(x_meth_train,y_train)
y_pred_meth = meth_pipeline.predict(x_meth_test)
y_prob_meth = meth_pipeline.predict_proba(x_meth_test)[:,1]

In [ ]:
print(y_pred_meth[:10])
print(y_prob_meth[:10])

### Evaluation of the model


In [ ]:
print("Accuracy:",accuracy_score(y_test,y_pred_meth))
print("precision:",precision_score(y_test,y_pred_meth))
print("Recall:",recall_score(y_test,y_pred_meth))
print("F1-score:",f1_score(y_test,y_pred_meth))
print("ROC_AUC:",roc_auc_score(y_test,y_prob_meth))
print("\nConfusion matrix:",confusion_matrix(y_test,y_pred_meth))
print("\nClassification report:",classification_report(y_test,y_pred_meth))

### Combined model


In [ ]:
#select 100 RNA and 100 methylation features seperately
x_combined_train = pd.concat([x_rna_train.add_prefix("RNA_"),x_meth_train.add_prefix("METH_")], axis=1)
x_combined_test = pd.concat([x_rna_test.add_prefix("RNA_"),x_meth_test.add_prefix("METH_")], axis=1)

print(x_combined_train.shape)
print(x_combined_test.shape)

In [ ]:
from sklearn.compose import ColumnTransformer, make_column_selector

In [ ]:
combined_preprocessing = ColumnTransformer([("rna", Pipeline([("variance", VarianceThreshold()),
                                                              ("selection", SelectKBest(score_func=f_classif, k=100)),
                                                              ("scaling", StandardScaler())]), 
                                                    make_column_selector(pattern="^RNA_")),
                                            ("meth", Pipeline([("imputer", SimpleImputer(strategy="median")),
                                                               ("variance", VarianceThreshold()),
                                                               ("selection", SelectKBest(score_func=f_classif, k=100)),
                                                               ("scaling", StandardScaler())]), 
                                                    make_column_selector(pattern="^METH_"))])

combined_pipeline = Pipeline([("preprocessing", combined_preprocessing),
                              ("classifier", LogisticRegression(class_weight="balanced",max_iter=2000,random_state=42))])

In [ ]:
combined_pipeline.fit(x_combined_train, y_train)
y_pred_combined = combined_pipeline.predict(x_combined_test)
y_prob_combined = combined_pipeline.predict_proba(x_combined_test)[:, 1]

In [ ]:
print(y_pred_combined[:10])
print(y_prob_combined[:10])

### Evaluation of the model

In [ ]:
print("Accuracy:",accuracy_score(y_test,y_pred_combined))
print("precision:",precision_score(y_test,y_pred_combined))
print("Recall:",recall_score(y_test,y_pred_combined))
print("F1-score:",f1_score(y_test,y_pred_combined))
print("ROC_AUC:",roc_auc_score(y_test,y_prob_combined))
print("\nConfusion matrix:",confusion_matrix(y_test,y_pred_combined))
print("\nClassification report:",classification_report(y_test,y_pred_combined))

## Cross validation


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [ ]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring = {"roc_auc":"roc_auc", "balanced_accuracy":"balanced_accuracy",
           "f1": "f1","precision": "precision","recall": "recall"}

In [ ]:
#RNA model
rna_cv =cross_validate(rna_pipeline,x_rna_train,y_train,cv=cv,scoring=scoring,n_jobs=-1)

#methylation model
meth_cv =cross_validate(meth_pipeline,x_meth_train,y_train,cv=cv,scoring=scoring,n_jobs=-1)

#combined model
combined_cv =cross_validate(combined_pipeline,x_combined_train,y_train,cv=cv,scoring=scoring,n_jobs=-1)


In [ ]:
#summary
models = [rna_cv,meth_cv,combined_cv]
names = ["RNA-only", "Methylation-only", "Combined"]

for name,m in zip(names,models):
    print(f"{name}")
    for metric in scoring:
        scores = m[f"test_{metric}"]
        print(f"{metric}: "
              f"{scores.mean():.3f} "
              f"+/- {scores.std():.3f}")
    print("\n")

In [ ]:
cv_summary = pd.DataFrame({"RNA-only": {metric: f"{rna_cv[f'test_{metric}'].mean():.3f} +/- "f"{rna_cv[f'test_{metric}'].std():.3f}"for metric in scoring},
    "Methylation-only": {metric: f"{meth_cv[f'test_{metric}'].mean():.3f} +/- "f"{meth_cv[f'test_{metric}'].std():.3f}"for metric in scoring},
    "Combined": {metric: f"{combined_cv[f'test_{metric}'].mean():.3f} +/- "f"{combined_cv[f'test_{metric}'].std():.3f}"for metric in scoring}})

In [ ]:
cv_summary


### test set comparison

In [ ]:
from sklearn.metrics import balanced_accuracy_score

In [ ]:
predictions = {"RNA-only": (y_pred_rna, y_prob_rna),
               "Methylation-only": (y_pred_meth, y_prob_meth),
               "Combined": (y_pred_combined, y_prob_combined)}

for name, (pred, prob) in predictions.items():
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred))
    print("F1:", f1_score(y_test, pred))
    print("ROC-AUC:", roc_auc_score(y_test, prob))

## Biological Interpretation

### RNA model

In [ ]:
#original feature names
rna_features = x_rna_train.columns

#feature mask
variance_mask = rna_pipeline.named_steps["variance"].get_support()
rna_features_after_variance = rna_features[variance_mask]

#features selected by SelectKBest
selection_mask = rna_pipeline.named_steps["selection"].get_support()

#Extract selected genes
selected_rna_genes = rna_features_after_variance[selection_mask]

print("Features before variance filtering:", len(rna_features))
print("Features after variance filtering:", len(rna_features_after_variance))
print("Selected RNA genes:", len(selected_rna_genes))
print(selected_rna_genes.tolist())

### Methylation model

In [ ]:
# methylation feature names
meth_features = x_meth_train.columns

# Features remaining after imputation
meth_features_after_imputation = (meth_pipeline.named_steps["imputer"].get_feature_names_out(meth_features))

# Features remaining after variance filtering
variance_mask_meth = (meth_pipeline.named_steps["variance"].get_support())

meth_features_after_variance = (meth_features_after_imputation[variance_mask_meth])

# Features selected by SelectKBest
selection_mask_meth = (meth_pipeline.named_steps["selection"].get_support())

selected_meth_probes = (meth_features_after_variance[selection_mask_meth])

print("Original methylation features:", len(meth_features))
print("After imputation:", len(meth_features_after_imputation))
print("After variance filtering:", len(meth_features_after_variance))
print("Selected methylation probes:", len(selected_meth_probes))

print(selected_meth_probes.tolist())

### Combined model

In [ ]:
# Access the fitted ColumnTransformer
preprocessor = combined_pipeline.named_steps["preprocessing"]

# Extract the RNA and methylation sub-pipelines
combined_rna_pipeline = preprocessor.named_transformers_["rna"]
combined_meth_pipeline = preprocessor.named_transformers_["meth"]

In [ ]:
combined_rna_features = x_combined_train.filter(regex="^RNA_").columns
rna_variance_mask = (combined_rna_pipeline.named_steps["variance"].get_support())
rna_features_after_variance = combined_rna_features[rna_variance_mask]
rna_selection_mask = (combined_rna_pipeline.named_steps["selection"].get_support())
selected_combined_rna = rna_features_after_variance[rna_selection_mask]

# Remove the RNA_ prefix
selected_combined_rna = selected_combined_rna.str.replace("^RNA_", "", regex=True)

print("Combined model RNA genes:", len(selected_combined_rna))
print(selected_combined_rna.tolist())

In [ ]:
combined_meth_features = x_combined_train.filter(regex="^METH_").columns
meth_variance_mask = (combined_meth_pipeline.named_steps["variance"].get_support())
meth_features_after_variance = combined_meth_features[meth_variance_mask]
meth_selection_mask = (combined_meth_pipeline.named_steps["selection"].get_support())
selected_combined_meth = meth_features_after_variance[meth_selection_mask]

# Remove the METH_ prefix
selected_combined_meth = selected_combined_meth.str.replace("^METH_", "", regex=True)

print("Combined model methylation probes:", len(selected_combined_meth))
print(selected_combined_meth.tolist())

In [ ]:
print(selected_combined_meth.tolist())

In [ ]:
#save the rna genes in a csv file
rna_genes_df = pd.DataFrame({"Gene": selected_rna_genes})
rna_genes_df.to_csv("selected_rna_genes.csv",index=False)

#save the probes in a csv file
probes_df = pd.DataFrame({"Probe_ID": selected_combined_meth})
probes_df.to_csv("selected_combined_meth_probes.csv",index=False)

## Methylation expression correlation analysis

In [ ]:
shared_probe_gene = {"cg03490200": "A2ML1","cg08887581": "C1orf64","cg24525573": "C1orf64",
                     "cg21488617": "DNALI1","cg10451565": "GPR77","cg19881895": "SLC43A3","cg02643667": "TFF1"}

print("RNA and methylation samples aligned:",x_rna_train.index.equals(x_meth_train.index))

for probe, gene in shared_probe_gene.items():
    print(probe,gene,"Probe available:", probe in x_meth_train.columns,
          "Gene available:", gene in x_rna_train.columns,
          "Missing methylation values:", x_meth_train[probe].isna().sum())

### Spearman correlation


In [ ]:
from scipy.stats import spearmanr

In [ ]:
correlation_results = []

for probe, gene in shared_probe_gene.items():

    # Extract the two variables and remove incomplete observations
    paired_data = pd.concat([x_meth_train[probe].rename("Methylation"),
                             x_rna_train[gene].rename("Expression")], axis=1).dropna()

    # Calculate Spearman correlation
    rho, p_value = spearmanr(paired_data["Methylation"],paired_data["Expression"])
    correlation_results.append({"Gene": gene,"Probe": probe,"N": len(paired_data),"Spearman_rho": rho,"p_value": p_value})

correlation_df = pd.DataFrame(correlation_results)
correlation_df = correlation_df.sort_values(by="Spearman_rho",key=lambda x: x.abs(),ascending=False)

print(correlation_df.to_string(index=False))

### Correct for multiple testing

In [ ]:
from statsmodels.stats.multitest import multipletests

In [ ]:
correlation_df["adjusted_p_value"] = multipletests(correlation_df["p_value"],method="fdr_bh")[1]
print(correlation_df[["Gene", "Probe", "Spearman_rho", "p_value", "adjusted_p_value"]].to_string(index=False))

### Plot the results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns 

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 13))
axes = axes.flatten()

for i, (probe, gene) in enumerate(shared_probe_gene.items()):
    paired_data = pd.concat([x_meth_train[probe].rename("Methylation"),
                             x_rna_train[gene].rename("Expression")],axis=1).dropna()

    rho = correlation_df.loc[correlation_df["Probe"] == probe,"Spearman_rho"].iloc[0]

    sns.regplot(data=paired_data, x="Methylation", y="Expression",ax=axes[i],scatter_kws={"alpha": 0.6},line_kws={"color": "red"})

    axes[i].set_title(f"{gene} ({probe})\nSpearman rho = {rho:.3f}")
    axes[i].set_xlabel("Methylation beta value")
    axes[i].set_ylabel("RNA expression")

# Remove the two unused panels
for j in range(len(shared_probe_gene), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### Colour the plots by ER status

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for i, (probe, gene) in enumerate(shared_probe_gene.items()):
    paired_data = pd.concat([x_meth_train[probe].rename("Methylation"),
                             x_rna_train[gene].rename("Expression"),
                             y_train.rename("ER_Status")],axis=1).dropna()

    paired_data["ER_Status"] = paired_data["ER_Status"].map({0: "ER-negative",1: "ER-positive"})

    sns.scatterplot(data=paired_data,x="Methylation",y="Expression",hue="ER_Status",alpha=0.65,ax=axes[i])

    axes[i].set_title(f"{gene} ({probe})")

    if i == 0:
        axes[i].legend(title="ER status")
    else:
        axes[i].get_legend().remove()

for j in range(len(shared_probe_gene), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
#spearmen correlation by ER status
group_correlation_results = []

for probe, gene in shared_probe_gene.items():
    paired_data = pd.concat([x_meth_train[probe].rename("Methylation"),
                             x_rna_train[gene].rename("Expression"),
                             y_train.rename("ER_Status")],axis=1).dropna()

    for status, label in [(1, "ER-positive"), (0, "ER-negative")]:
        group_data = paired_data[paired_data["ER_Status"] == status]
        rho, p_value = spearmanr(group_data["Methylation"],group_data["Expression"])

        group_correlation_results.append({
            "Gene": gene,
            "Probe": probe,
            "ER_group": label,
            "N": len(group_data),
            "Spearman_rho": rho,
            "p_value": p_value
        })

group_correlation_df = pd.DataFrame(group_correlation_results)

print(group_correlation_df.to_string(index=False))

## Improvements

In [82]:

# Split the original sample IDs
train_ids_v2, test_ids_v2 = train_test_split(
    y.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# RNA matrices
x_rna_train_v2 = x_rna.loc[train_ids_v2].copy()
x_rna_test_v2 = x_rna.loc[test_ids_v2].copy()

# Methylation matrices
x_meth_train_v2 = x_meth.loc[train_ids_v2].copy()
x_meth_test_v2 = x_meth.loc[test_ids_v2].copy()

# Target
y_train_v2 = y.loc[train_ids_v2]
y_test_v2 = y.loc[test_ids_v2]

# Check dimensions
print("RNA training:", x_rna_train_v2.shape)
print("RNA testing:", x_rna_test_v2.shape)

print("Methylation training:", x_meth_train_v2.shape)
print("Methylation testing:", x_meth_test_v2.shape)

print("Training labels:", y_train_v2.shape)
print("Testing labels:", y_test_v2.shape)

# Check alignment
print(
    "RNA and methylation aligned:",
    x_rna_train_v2.index.equals(x_meth_train_v2.index)
)

print(
    "RNA and target aligned:",
    x_rna_train_v2.index.equals(y_train_v2.index)
)

print(
    "Methylation and target aligned:",
    x_meth_train_v2.index.equals(y_train_v2.index)
)

print("Training class distribution:")
print(y_train_v2.value_counts())

print("Testing class distribution:")
print(y_test_v2.value_counts())

RNA training: (245, 20530)
RNA testing: (62, 20530)
Methylation training: (245, 27578)
Methylation testing: (62, 27578)
Training labels: (245,)
Testing labels: (62,)
RNA and methylation aligned: True
RNA and target aligned: True
Methylation and target aligned: True
Training class distribution:
ER_Status_nature2012
1    190
0     55
Name: count, dtype: int64
Testing class distribution:
ER_Status_nature2012
1    48
0    14
Name: count, dtype: int64


In [83]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class MissingnessFilter(BaseEstimator, TransformerMixin):

    def __init__(self, max_missing_fraction=0.05):
        self.max_missing_fraction = max_missing_fraction

    def fit(self, X, y=None):

        # Calculate missingness using the training data
        missing_fraction = X.isna().mean(axis=0)

        # Retain probes with <= 5% missingness
        self.support_ = (
            missing_fraction <= self.max_missing_fraction
        ).to_numpy()

        self.feature_names_in_ = np.asarray(X.columns)

        return self

    def transform(self, X):

        check_is_fitted(self, "support_")

        # Retain only the probes selected during fitting
        return X.loc[:, self.feature_names_in_[self.support_]]

    def get_feature_names_out(self, input_features=None):

        check_is_fitted(self, "support_")

        return self.feature_names_in_[self.support_]

In [84]:
meth_missing_filter = MissingnessFilter(max_missing_fraction=0.05)

# Fit using training samples only
x_meth_train_filtered_v2 = meth_missing_filter.fit_transform(
    x_meth_train_v2
)

# Apply the same filter to test samples
x_meth_test_filtered_v2 = meth_missing_filter.transform(
    x_meth_test_v2
)

print("Original training matrix:", x_meth_train_v2.shape)
print("Filtered training matrix:", x_meth_train_filtered_v2.shape)

print("Original test matrix:", x_meth_test_v2.shape)
print("Filtered test matrix:", x_meth_test_filtered_v2.shape)

print(
    "Remaining training probes with >5% missing:",
    (x_meth_train_filtered_v2.isna().mean() > 0.05).sum()
)

print(
    "Training and test columns aligned:",
    x_meth_train_filtered_v2.columns.equals(
        x_meth_test_filtered_v2.columns
    )
)

Original training matrix: (245, 27578)
Filtered training matrix: (245, 24941)
Original test matrix: (62, 27578)
Filtered test matrix: (62, 24941)
Remaining training probes with >5% missing: 0
Training and test columns aligned: True


In [85]:
rna_pipeline_v2 = Pipeline([
    ("variance", VarianceThreshold()),

    ("selection", SelectKBest(
        score_func=f_classif,
        k=100
    )),

    ("scaling", StandardScaler()),

    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

# Train
rna_pipeline_v2.fit(x_rna_train_v2, y_train_v2)

# Predict
y_pred_rna_v2 = rna_pipeline_v2.predict(x_rna_test_v2)
y_prob_rna_v2 = rna_pipeline_v2.predict_proba(x_rna_test_v2)[:, 1]

print("RNA pipeline successfully fitted.")

print(
    "Genes after variance filtering:",
    rna_pipeline_v2.named_steps["variance"].get_support().sum()
)

print(
    "Selected genes:",
    rna_pipeline_v2.named_steps["selection"].get_support().sum()
)

RNA pipeline successfully fitted.
Genes after variance filtering: 20149
Selected genes: 100


In [86]:
meth_pipeline_v2 = Pipeline([
    ("missingness", MissingnessFilter(max_missing_fraction=0.05)),

    ("imputer", SimpleImputer(strategy="median")),

    ("variance", VarianceThreshold()),

    ("selection", SelectKBest(
        score_func=f_classif,
        k=100
    )),

    ("scaling", StandardScaler()),

    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

# Train
meth_pipeline_v2.fit(x_meth_train_v2, y_train_v2)

# Predict
y_pred_meth_v2 = meth_pipeline_v2.predict(x_meth_test_v2)
y_prob_meth_v2 = meth_pipeline_v2.predict_proba(x_meth_test_v2)[:, 1]

print("Methylation pipeline successfully fitted.")

print(
    "Probes after missingness filtering:",
    len(meth_pipeline_v2.named_steps["missingness"].get_feature_names_out())
)

print(
    "Probes after variance filtering:",
    meth_pipeline_v2.named_steps["variance"].get_support().sum()
)

print(
    "Selected probes:",
    meth_pipeline_v2.named_steps["selection"].get_support().sum()
)

Methylation pipeline successfully fitted.
Probes after missingness filtering: 24941
Probes after variance filtering: 24941
Selected probes: 100


In [87]:
# Combine the original training matrices
x_combined_train_v2 = pd.concat([
    x_rna_train_v2.add_prefix("RNA_"),
    x_meth_train_v2.add_prefix("METH_")
], axis=1)

# Combine the original test matrices
x_combined_test_v2 = pd.concat([
    x_rna_test_v2.add_prefix("RNA_"),
    x_meth_test_v2.add_prefix("METH_")
], axis=1)

# Check dimensions and alignment
print("Combined training set:", x_combined_train_v2.shape)
print("Combined test set:", x_combined_test_v2.shape)

print(
    "Training samples aligned:",
    x_combined_train_v2.index.equals(y_train_v2.index)
)

print(
    "Test samples aligned:",
    x_combined_test_v2.index.equals(y_test_v2.index)
)

Combined training set: (245, 48108)
Combined test set: (62, 48108)
Training samples aligned: True
Test samples aligned: True


In [88]:
combined_preprocessing_v2 = ColumnTransformer([

    # RNA branch
    ("rna", Pipeline([
        ("variance", VarianceThreshold()),
        ("selection", SelectKBest(score_func=f_classif, k=100)),
        ("scaling", StandardScaler())
    ]), make_column_selector(pattern="^RNA_")),

    # Methylation branch
    ("meth", Pipeline([
        ("missingness", MissingnessFilter(max_missing_fraction=0.05)),
        ("imputer", SimpleImputer(strategy="median")),
        ("variance", VarianceThreshold()),
        ("selection", SelectKBest(score_func=f_classif, k=100)),
        ("scaling", StandardScaler())
    ]), make_column_selector(pattern="^METH_"))
])

In [89]:
combined_pipeline_v2 = Pipeline([
    ("preprocessing", combined_preprocessing_v2),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

# Fit using training data only
combined_pipeline_v2.fit(
    x_combined_train_v2,
    y_train_v2
)

print("Combined model successfully fitted!")

Combined model successfully fitted!


In [90]:
# Predicted classes
y_pred_combined_v2 = combined_pipeline_v2.predict(
    x_combined_test_v2
)

# Predicted probabilities for the positive class
y_prob_combined_v2 = combined_pipeline_v2.predict_proba(
    x_combined_test_v2
)[:, 1]

print("Predicted classes:", y_pred_combined_v2[:10])
print("Predicted probabilities:", y_prob_combined_v2[:10])

Predicted classes: [1 1 1 1 0 1 0 1 0 1]
Predicted probabilities: [0.98368511 0.99963042 0.97565914 0.99986851 0.22541909 0.99965068
 0.02869828 0.99997077 0.01208427 0.99997525]


In [91]:
models_v2 = {
    "RNA-only": (y_pred_rna_v2, y_prob_rna_v2),
    "Methylation-only": (y_pred_meth_v2, y_prob_meth_v2),
    "Combined": (y_pred_combined_v2, y_prob_combined_v2)
}

results_v2 = []

for name, (y_pred, y_prob) in models_v2.items():

    results_v2.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_v2, y_pred),
        "Precision": precision_score(y_test_v2, y_pred),
        "Recall": recall_score(y_test_v2, y_pred),
        "F1-score": f1_score(y_test_v2, y_pred),
        "ROC-AUC": roc_auc_score(y_test_v2, y_prob),
        "Balanced accuracy": balanced_accuracy_score(y_test_v2, y_pred)
    })

results_v2_df = pd.DataFrame(results_v2).set_index("Model")

print(results_v2_df.round(4))

                  Accuracy  Precision  Recall  F1-score  ROC-AUC  \
Model                                                              
RNA-only            0.9677     0.9792  0.9792    0.9792   0.9940   
Methylation-only    0.9032     0.9200  0.9583    0.9388   0.9881   
Combined            0.9516     0.9787  0.9583    0.9684   0.9955   

                  Balanced accuracy  
Model                                
RNA-only                     0.9539  
Methylation-only             0.8363  
Combined                     0.9435  


In [92]:
for name, (y_pred, y_prob) in models_v2.items():

    print(f"\n{name}")
    print(confusion_matrix(y_test_v2, y_pred))


RNA-only
[[13  1]
 [ 1 47]]

Methylation-only
[[10  4]
 [ 2 46]]

Combined
[[13  1]
 [ 2 46]]


In [95]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Define evaluation metrics
scoring = {
    "roc_auc": "roc_auc",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}
pipelines_v2 = {
    "RNA-only": (rna_pipeline_v2, x_rna_train_v2),
    "Methylation-only": (meth_pipeline_v2, x_meth_train_v2),
    "Combined": (combined_pipeline_v2, x_combined_train_v2)
}

# Run cross-validation
cv_results_v2 = {}

for name, (pipeline, X_train) in pipelines_v2.items():

    print(f"Running cross-validation for {name}...")

    scores = cross_validate(
        pipeline,
        X_train,
        y_train_v2,
        cv=cv,
        scoring=scoring,
        n_jobs=1
    )

    cv_results_v2[name] = {
        metric: (
            scores[f"test_{metric}"].mean(),
            scores[f"test_{metric}"].std()
        )
        for metric in scoring
    }

print("Cross-validation completed!")

cv_summary_v2 = []

for model, metrics in cv_results_v2.items():

    row = {"Model": model}

    for metric, (mean, std) in metrics.items():
        row[f"{metric} (mean)"] = mean
        row[f"{metric} (std)"] = std

    cv_summary_v2.append(row)

cv_summary_v2_df = pd.DataFrame(cv_summary_v2).set_index("Model")

print(cv_summary_v2_df.round(4))

Running cross-validation for RNA-only...
Running cross-validation for Methylation-only...
Running cross-validation for Combined...
Cross-validation completed!
                  roc_auc (mean)  roc_auc (std)  balanced_accuracy (mean)  \
Model                                                                       
RNA-only                  0.9766         0.0141                    0.8919   
Methylation-only          0.9684         0.0135                    0.9036   
Combined                  0.9780         0.0094                    0.9361   

                  balanced_accuracy (std)  f1 (mean)  f1 (std)  \
Model                                                            
RNA-only                           0.0575     0.9499    0.0152   
Methylation-only                   0.0335     0.9549    0.0188   
Combined                           0.0319     0.9680    0.0143   

                  precision (mean)  precision (std)  recall (mean)  \
Model                                                 

In [96]:
# Get original gene names
rna_features_v2 = x_rna_train_v2.columns

# Features retained by variance filtering
rna_variance_mask_v2 = (
    rna_pipeline_v2.named_steps["variance"].get_support()
)

rna_features_after_variance_v2 = (
    rna_features_v2[rna_variance_mask_v2]
)

# Features retained by SelectKBest
rna_selection_mask_v2 = (
    rna_pipeline_v2.named_steps["selection"].get_support()
)

selected_rna_genes_v2 = (
    rna_features_after_variance_v2[rna_selection_mask_v2]
)

print("Selected RNA genes:", len(selected_rna_genes_v2))
print(selected_rna_genes_v2.tolist())
# Get the probes retained by missingness filtering
meth_features_after_missingness_v2 = (
    meth_pipeline_v2.named_steps["missingness"].get_feature_names_out()
)

# Features retained by variance filtering
meth_variance_mask_v2 = (
    meth_pipeline_v2.named_steps["variance"].get_support()
)

meth_features_after_variance_v2 = (
    meth_features_after_missingness_v2[meth_variance_mask_v2]
)

# Features retained by SelectKBest
meth_selection_mask_v2 = (
    meth_pipeline_v2.named_steps["selection"].get_support()
)

selected_meth_probes_v2 = (
    meth_features_after_variance_v2[meth_selection_mask_v2]
)

print("Selected methylation probes:", len(selected_meth_probes_v2))
print(selected_meth_probes_v2.tolist())
# RNA comparison
print(
    "RNA features identical:",
    set(selected_rna_genes) == set(selected_rna_genes_v2)
)

print(
    "RNA shared features:",
    len(set(selected_rna_genes) & set(selected_rna_genes_v2))
)

# Methylation comparison
print(
    "Methylation features identical:",
    set(selected_meth_probes) == set(selected_meth_probes_v2)
)

print(
    "Methylation shared features:",
    len(set(selected_meth_probes) & set(selected_meth_probes_v2))
)

Selected RNA genes: 100
['THSD4', 'ZNF703', 'ART3', 'ABAT', 'GPR77', 'TBC1D9', 'RAET1L', 'FSIP1', 'B3GNT5', 'SLC43A3', 'RABEP1', 'ZNF552', 'SYTL4', 'TLE3', 'NAT1', 'INPP4B', 'FLJ45983', 'CCNE1', 'MLPH', 'FAM174A', 'C6orf211', 'C10orf32', 'EVL', 'CA12', 'AGR2', 'AGR3', 'SLC39A12', 'MGC2889', 'CDK2AP1', 'GAMT', 'SFRS13B', 'P4HTM', 'EXOC6', 'RERG', 'AFF3', 'PRSS33', 'PPP1R14C', 'SLC39A6', 'WWP1', 'SCUBE2', 'TTC36', 'ARL3', 'KCNJ11', 'KCMF1', 'A2ML1', 'ESR1', 'MAPT', 'CT62', 'CACNA2D2', 'XBP1', 'DNALI1', 'CELSR1', 'SLC15A1', 'PRR15', 'IFRD1', 'ERBB4', 'CLSTN2', 'PADI2', 'ANXA9', 'KCNN4', 'C9orf170', 'ROPN1', 'C1orf64', 'C6orf97', 'POU5F1', 'MYB', 'DEGS2', 'APBB2', 'GATA3', 'FBP1', 'HSD17B4', 'ACADSB', 'FOXA1', 'ROPN1B', 'DACH1', 'CXXC5', 'KCNK15', 'KDM4B', 'CYP2B7P1', 'FZD9', 'SERPINA11', 'C5orf30', 'PCP2', 'AMY1A', 'CXorf61', 'TFF1', 'ARSG', 'CENPW', 'YBX1', 'GREB1', 'VGLL1', 'BCL11A', 'ANKRA2', 'PSAT1', 'ZMYND10', 'NCK2', 'YEATS2', 'LOC145837', 'SLC22A5', 'ABCC8']
Selected methylation pr